In [1]:
cd /workspace/hardware_rounding_error_predictor
git log --oneline -3
# Top should be: 95f7328 Patch flash_attn BEFORE loading the model

7220105 (HEAD -> claude/forward-pass-script-UsTJU, origin/claude/forward-pass-script-UsTJU) Patch flash_attn.flash_attn_func instead of _flash_attn_forward
bde1d2d Record GPU name in meta.json so CPU-only emulation works
93c476d Fix two real bugs from second-pass review


In [2]:
git pull
git log --oneline -3
# Top should now be: 95f7328 Patch flash_attn BEFORE loading the model

remote: Enumerating objects: 5, done.        
remote: Counting objects: 100% (5/5), done.        
remote: Compressing objects: 100% (1/1), done.        
remote: Total 3 (delta 2), reused 3 (delta 2), pack-reused 0 (from 0)        
Unpacking objects: 100% (3/3), 1.82 KiB | 621.00 KiB/s, done.
From https://github.com/NaciCankaya/hardware_rounding_error_predictor
   7220105..95f7328  claude/forward-pass-script-UsTJU -> origin/claude/forward-pass-script-UsTJU
Updating 7220105..95f7328
Fast-forward
 capture_forward.py | 117 ++++++++++++++++++++++++++++-------------------------
 1 file changed, 62 insertions(+), 55 deletions(-)
95f7328 (HEAD -> claude/forward-pass-script-UsTJU, origin/claude/forward-pass-script-UsTJU) Patch flash_attn BEFORE loading the model
7220105 Patch flash_attn.flash_attn_func instead of _flash_attn_forward
bde1d2d Record GPU name in meta.json so CPU-only emulation works


In [3]:
rm -rf forward_capture_smoke
python3 -u capture_forward.py 32 forward_capture_smoke

FULL FORWARD PASS CAPTURE
  model:   Qwen/Qwen3-4B
  max_seq: 32
  out_dir: forward_capture_smoke

Loading config for Qwen/Qwen3-4B...
  flash_attn.flash_attn_func monkey-patched for 36 layers (pre-model-load)
Loading Qwen/Qwen3-4B...
`torch_dtype` is deprecated! Use `dtype` instead!
Loading weights: 100%|███████████████████████| 398/398 [00:01<00:00, 220.66it/s]
  36 layers, hidden=2560, ffn=9728
  heads=32Q/8KV, head_dim=128, GQA=4
  has_qk_norm=True, rope_theta=1000000, rope_type=default

TOKENIZATION
------------------------------------------------------------
  Using PDF (PyPDF2): /workspace/hardware_rounding_error_predictor/literature/2403.00232v1.pdf
  Sequence length: 32 tokens

REGISTERING HOOKS
------------------------------------------------------------
  576 hooks registered across 36 layers

RUNNING FORWARD PASS...
done.

  flash_attn patch fired 36 times (expected ≥36)
COMPUTING RoPE cos/sin (GPU)...
  rope_cos_gpu.npy  (32, 128)
  rope_sin_gpu.npy  (32, 128)

SAVING EMBE

In [4]:
python3 -u -c "from mufu_emulator import MUFUEmulator; MUFUEmulator(gpu_name='A100')"

MUFU.RSQ: no cache for A100, probing hardware...
  even (biased_exp=127): 6505163/8388608 exact, ±1: 1883437, ±2: 8
  odd (biased_exp=128): 6637923/8388608 exact, ±1: 1750685, ±2: 0
MUFU.RSQ: cached to mufu_cache/A100/
MUFU.EX2: generating full table for A100 (4GB, ~60s)...
  Compiled --use_fast_math exp2 kernel
    12% (8s, ETA 58s)
    25% (17s, ETA 50s)
/workspace/hardware_rounding_error_predictor/mufu_emulator.py:476: RuntimeWarning: overflow encountered in cast
  exact = np.exp2(float_vals.astype(np.float64)).astype(np.float32)
/workspace/hardware_rounding_error_predictor/mufu_emulator.py:476: RuntimeWarning: overflow encountered in exp2
  exact = np.exp2(float_vals.astype(np.float64)).astype(np.float32)
    38% (27s, ETA 45s)
/workspace/hardware_rounding_error_predictor/mufu_emulator.py:476: RuntimeWarning: invalid value encountered in cast
  exact = np.exp2(float_vals.astype(np.float64)).astype(np.float32)
    50% (38s, ETA 38s)
    62% (47s, ETA 28s)
    75% (55s, ETA 18s)
    

In [5]:
tar czf capture_smoke.tar.gz forward_capture_smoke/ mufu_cache/
ls -lh capture_smoke.tar.gz

-rw-rw-r-- 1 root root 197M Apr 21 01:37 capture_smoke.tar.gz


In [6]:
python3 -u emulate_forward.py forward_capture_smoke

FULL FORWARD PASS EMULATION
  capture: forward_capture_smoke
  model:   Qwen/Qwen3-4B
  seq_len: 32  layers: 36  hidden: 2560

MUFU.RSQ: loaded cached tables for A100
MUFU.EX2: loaded full table for A100 (4.3GB)
MUFU.RCP: loaded cached table for A100
Emulating A100  —  A100 bf16→fp32: NFMA=8, neab=1, window=(2,23,1)=26b, round=trunc, denorm=True, MMA=(16, 8, 16), c_order=early, interleaved=False

Loading Qwen/Qwen3-4B weights on CPU (this takes a moment)...
`torch_dtype` is deprecated! Use `dtype` instead!
Loading weights: 100%|██████████████████████| 398/398 [00:00<00:00, 4670.16it/s]
done.

Loading RoPE cos/sin...
  cos/sin shape: (32, 128)

LAYER EMULATION
------------------------------------------------------------
   Layer          Attn           FFN      Time
  ----------------------------------------------
Traceback (most recent call last):
  File "/workspace/hardware_rounding_error_predictor/emulate_forward.py", line 398, in <module>
    emulate(cap_dir=cap_dir)
  File "/worksp

: 1

In [7]:
git pull
python3 -u emulate_forward.py forward_capture_smoke

remote: Enumerating objects: 5, done.        
remote: Counting objects: 100% (5/5), done.        
remote: Compressing objects: 100% (1/1), done.        
remote: Total 3 (delta 2), reused 3 (delta 2), pack-reused 0 (from 0)        
Unpacking objects: 100% (3/3), 1.27 KiB | 434.00 KiB/s, done.
From https://github.com/NaciCankaya/hardware_rounding_error_predictor
   95f7328..1c8f5d3  claude/forward-pass-script-UsTJU -> origin/claude/forward-pass-script-UsTJU
Updating 95f7328..1c8f5d3
Fast-forward
 emulate_forward.py | 36 +++++++++++++++++++++++-------------
 1 file changed, 23 insertions(+), 13 deletions(-)
FULL FORWARD PASS EMULATION
  capture: forward_capture_smoke
  model:   Qwen/Qwen3-4B
  seq_len: 32  layers: 36  hidden: 2560

MUFU.RSQ: loaded cached tables for A100
MUFU.EX2: loaded full table for A100 (4.3GB)
MUFU.RCP: loaded cached table for A100
Emulating A100  —  A100 bf16→fp32: NFMA=8, neab=1, window=(2,23,1)=26b, round=trunc, denorm=True, MMA=(16, 8, 16), c_order=early, interle

: 1

In [9]:
cd /workspace

# Clone CUTLASS headers (shallow, we don't need full history)
git clone --depth 1 --branch v3.5.0 https://github.com/NVIDIA/cutlass.git

# Build the binary (back in the repo dir)
cd /workspace/hardware_rounding_error_predictor
nvcc -o cutlass_gemm_flex cutlass_gemm_flex.cu \
  -I /workspace/cutlass/include \
  -I /workspace/cutlass/tools/util/include \
  -arch=sm_80 -std=c++17 -O2

# Verify
ls -lh cutlass_gemm_flex
./cutlass_gemm_flex 2>&1 | head -5

Cloning into 'cutlass'...
remote: Enumerating objects: 5992, done.        
remote: Counting objects: 100% (5992/5992), done.        
remote: Compressing objects: 100% (1638/1638), done.        
remote: Total 5992 (delta 3485), reused 4944 (delta 3066), pack-reused 0 (from 0)        
Receiving objects: 100% (5992/5992), 27.23 MiB | 9.92 MiB/s, done.
Resolving deltas: 100% (3485/3485), done.
Note: switching to '7d49e6c7e2f8896c47f586706e67e1fb215529dc'.

You are in 'detached HEAD' state. You can look around, make experimental
changes and commit them, and you can discard any commits you make in this
state without impacting any branches by switching back to a branch.

If you want to create a new branch to retain commits you create, you may
do so (now or later) by using -c with the switch command. Example:

  git switch -c <new-branch-name>

Or undo this operation with:

  git switch -

Turn off this advice by setting config variable advice.detachedHead to false

cutlass_gemm_flex.cu(165): 

In [10]:
rm -rf forward_capture_smoke
python3 -u capture_forward.py 32 forward_capture_smoke

CUTLASS-CONSISTENT FORWARD PASS CAPTURE
  model:   Qwen/Qwen3-4B
  max_seq: 32
  out_dir: forward_capture_smoke

Loading Qwen/Qwen3-4B...
`torch_dtype` is deprecated! Use `dtype` instead!
Loading weights: 100%|███████████████████████| 398/398 [00:01<00:00, 296.66it/s]
  36 layers, hidden=2560, ffn=9728
  heads=32Q/8KV, head_dim=128, GQA=4
  has_qk_norm=True, rope_theta=1000000

TOKENIZATION
------------------------------------------------------------
  Using PDF (PyPDF2): /workspace/hardware_rounding_error_predictor/literature/2403.00232v1.pdf
  Sequence length: 32 tokens

COMPUTING RoPE cos/sin (GPU)...
  rope_cos_gpu.npy / rope_sin_gpu.npy saved

EMBEDDING
------------------------------------------------------------
  embed_out.bin  [32, 2560]

LAYER-BY-LAYER FORWARD PASS
------------------------------------------------------------
  CUTLASS tempdir: /tmp/cutlass_capture_hjt2jygm

  Layer  0/35: done (4.6s, cumulative CUTLASS calls 7)
  Layer  5/35: done (4.5s, cumulative CUTLASS cal

In [11]:
python3 -u emulate_forward.py forward_capture_smoke

FULL FORWARD PASS EMULATION
  capture: forward_capture_smoke
  model:   Qwen/Qwen3-4B
  seq_len: 32  layers: 36  hidden: 2560

MUFU.RSQ: loaded cached tables for A100
MUFU.EX2: loaded full table for A100 (4.3GB)
MUFU.RCP: loaded cached table for A100
Emulating A100  —  A100 bf16→fp32: NFMA=8, neab=1, window=(2,23,1)=26b, round=trunc, denorm=True, MMA=(16, 8, 16), c_order=early, interleaved=False

Loading Qwen/Qwen3-4B weights on CPU (this takes a moment)...
`torch_dtype` is deprecated! Use `dtype` instead!
Loading weights: 100%|██████████████████████| 398/398 [00:00<00:00, 8079.75it/s]
done.

Loading RoPE cos/sin...
  cos/sin shape: (32, 128)

LAYER EMULATION
------------------------------------------------------------
   Layer          Attn           FFN      Time
  ----------------------------------------------

  Layer  0 attn:
  FA2 attention core...
    Head 0/32...
    Head 8/32 (0s, ETA 0s)...
    Head 16/32 (0s, ETA 0s)...
    Head 24/32 (0s, ETA 0s)...
    FA2 core done (0.3s)